Проверенная редакция 18.11.2026. Исторические результаты сохранены в results/. Корпус и выборки фиксированы. Новые запуски пишут в отдельные рабочие каталоги. Проверка: scripts/audit_release.py. Полные модельные прогоны в этой редакции не повторялись.

# Lossy-RAG — Финальная проверка устойчивости к перефразированию

**Это самостоятельный ноутбук.** Для запуска на том же компьютере: **Kernel → Restart & Run All**. Никаких ручных изменений путей и повторного кодирования 82 222 документов.

**Цель:** сравнить BGE Small / Base / Large на **одних и тех же 500 парах**: исходный вопрос и его перефраз, с неизменным эталоном правильных фрагментов и полным корпусом SQuAD. Мы сравниваем исходную/перефразированную формулировки **только внутри пар**, а не с прежними метриками для другой случайной выборки.

**Что делает:**
1. Находит подготовленные CSV в `D:/Lossy-RAG/data` и готовые документные эмбеддинги из `lossy_rag_final_results_safe/cache`.
2. Выбирает **500 реальных пар**, для которых существуют оригинал и перефраз, и проверяет соответствие эталонным документам.
3. Кодирует **только 1 000 вопросов для каждой модели**; документные эмбеддинги повторно **не вычисляются**.
4. Считает строгий и мягкий Hit@1, 3, 5, 10 точным косинусным поиском; сохраняет результаты **после каждой модели**.
5. Рассчитывает парные изменения, доверительные интервалы, точный тест Мак-Немара и поправку Холма, строит графики.

**Если нужный кэш документов отсутствует, ноутбук остановится с понятной ошибкой, а не начнёт ещё одну многочасовую векторизацию.** Если число пригодных пар меньше 500, тоже будет объяснение.

Выход: `D:/Lossy-RAG/lossy_rag_paraphrase_results/`. Существующие результаты BGE, PCA и Nomic не изменяются.

In [ ]:
# 0. Пути относительно текущего проекта; поддерживается LOSSY_RAG_ROOT.
import os
import sys
from pathlib import Path

# Root is explicit, or the nearest repository above the notebook working directory.
_candidates = [Path.cwd(), *Path.cwd().parents]
_detected = next((p for p in _candidates if (p / 'data/build_data.py').is_file()), None)
_value = os.environ.get('LOSSY_RAG_ROOT')
if not _value and _detected is None:
    raise RuntimeError('Open Jupyter inside Lossy-RAG or set LOSSY_RAG_ROOT to the repository root.')
ROOT = Path(_value).expanduser().resolve() if _value else _detected
BLOCK1_DIR = ROOT / 'experiments/block1'
sys.path.insert(0, str(BLOCK1_DIR / 'scripts'))
from retrieval import array_hash, stable_topk_positions

DATA_DIR = ROOT / "data"
OLD_RESULTS = ROOT / "lossy_rag_final_results_safe"
OUTPUT_DIR = ROOT / "lossy_rag_paraphrase_results"
for name in ("cache", "rankings", "tables", "figures"):
    (OUTPUT_DIR / name).mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(ROOT / "hf_cache")
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "15"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
N_PAIRS = 500
MIN_CORPUS_ROWS = 80_000
K_VALUES = (1, 3, 5, 10)
BATCH_SIZE = 16
QUERY_BLOCK = 16
DOC_BLOCK = 4096
THREADS = 4
DEVICE = "auto"

MODELS = {
    "bge_small": ("BAAI/bge-small-en-v1.5", 384),
    "bge_base": ("BAAI/bge-base-en-v1.5", 768),
    "bge_large": ("BAAI/bge-large-en-v1.5", 1024),
}
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
print("Источник данных:", DATA_DIR)
print("Готовые документные эмбеддинги:", OLD_RESULTS / "cache")
print("Новые результаты:", OUTPUT_DIR)

In [ ]:
# 1. Библиотеки. Устанавливаем только отсутствующие в текущем ядре Jupyter.
import importlib.util
import subprocess

required = {
    "numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
    "matplotlib": "matplotlib", "threadpoolctl": "threadpoolctl",
    "sentence_transformers": "sentence-transformers", "torch": "torch",
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Устанавливаю отсутствующие библиотеки:", missing, flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
    importlib.invalidate_caches()

import gc
import hashlib
import json
import time
from collections import defaultdict
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from threadpoolctl import threadpool_limits

try:
    import torch
    torch.set_num_threads(max(1, THREADS))
    DEVICE_USED = ("cuda" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else DEVICE
except ImportError:
    DEVICE_USED = "cpu"

LOG_FILE = OUTPUT_DIR / "paraphrase_run_log.txt"
def log(message):
    line = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}"
    print(line, flush=True)
    with LOG_FILE.open("a", encoding="utf-8") as f:
        f.write(line + "\n")

def atomic_json(path, obj):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary.replace(path)

def signature(ids, texts):
    # Точная копия алгоритма fingerprint из уже выполненного эксперимента.
    h = hashlib.sha256()
    for id_, text in zip(ids, texts):
        h.update(str(int(id_)).encode())
        h.update(b"\0")
        h.update(str(text).encode("utf-8"))
        h.update(b"\n")
    return h.hexdigest()

def normalized(a):
    a = np.asarray(a, dtype=np.float32)
    return a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-12)

def bool_column(s):
    return s.astype(str).str.strip().str.lower().isin(["1", "true", "yes"])

log(f"Запуск 500 пар; Python={sys.version.split()[0]}; device={DEVICE_USED}")

In [ ]:
# 2. Читаем готовый корпус и выбираем ТОЛЬКО вопросы с существующими перефразами.
required_files = ("chunks.csv", "questions.csv", "answer_chunks.csv")
missing_files = [x for x in required_files if not (DATA_DIR / x).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Нет {missing_files} в {DATA_DIR}. Не запускаем сборку: "
        "сначала проверьте расположение результатов предыдущего эксперимента."
    )
chunks = pd.read_csv(DATA_DIR / "chunks.csv")
questions = pd.read_csv(DATA_DIR / "questions.csv")
answer_chunks = pd.read_csv(DATA_DIR / "answer_chunks.csv")
for name, frame, required_cols in [
    ("chunks", chunks, ("chunk_id", "text")),
    ("questions", questions, ("qid", "orig_qid", "role", "question")),
    ("answer_chunks", answer_chunks, ("orig_qid", "chunk_id", "is_annotated")),
]:
    if not set(required_cols).issubset(frame.columns):
        raise ValueError(f"Неверная структура {name}: отсутствуют {set(required_cols)-set(frame.columns)}")
if len(chunks) < MIN_CORPUS_ROWS:
    raise ValueError(f"Найдено {len(chunks)} фрагментов вместо полного корпуса. Остановлено.")
chunks["chunk_id"] = chunks["chunk_id"].astype("int64")
questions["qid"] = questions["qid"].astype("int64")
questions["orig_qid"] = questions["orig_qid"].astype("int64")
answer_chunks["orig_qid"] = answer_chunks["orig_qid"].astype("int64")
answer_chunks["chunk_id"] = answer_chunks["chunk_id"].astype("int64")
answer_chunks["is_annotated"] = bool_column(answer_chunks["is_annotated"])
questions["is_paraphrase"] = (bool_column(questions["is_paraphrase"])
                              if "is_paraphrase" in questions.columns
                              else questions["qid"] != questions["orig_qid"])
if not chunks.chunk_id.is_unique or not questions.qid.is_unique:
    raise ValueError("Обнаружены повторяющиеся chunk_id или qid")
if chunks.text.isna().any() or questions.question.isna().any():
    raise ValueError("В данных присутствуют пустые тексты документов или вопросов")

# Если в questions.csv почему-то нет 500 перефразов, подключаем отдельный CSV
# из того же проекта. Не генерируем искусственные перефразы.
train_orig = questions[(questions.role.astype(str).str.lower() == "train") &
                       (~questions.is_paraphrase)].copy()
par = questions[(questions.role.astype(str).str.lower() == "train") &
                (questions.is_paraphrase)].copy()
if par.orig_qid.nunique() < N_PAIRS:
    raw_file = DATA_DIR / "questions_paraphrased.csv"
    if not raw_file.exists():
        from urllib.request import Request, urlopen
        url = "https://raw.githubusercontent.com/Arnoochka/Lossy-RAG/198aa66b0003ef759ecca6cd779176913df02fbc/data/questions_paraphrased.csv"
        log("Отдельный CSV перефраз не найден; пробую получить файл из репозитория проекта")
        try:
            req = Request(url, headers={"User-Agent": "Lossy-RAG-paraphrase-experiment"})
            with urlopen(req, timeout=120) as response:
                data = response.read(30_000_001)
            if not data or len(data) > 30_000_000 or b"orig_qid" not in data[:4096]:
                raise ValueError("Некорректное содержимое CSV перефраз")
            raw_file.write_bytes(data)
        except Exception as exc:
            raise RuntimeError(
                "Не удалось загрузить questions_paraphrased.csv. "
                "Положите оригинальный CSV из репозитория в папку D:/Lossy-RAG/data."
            ) from exc
    extra = pd.read_csv(raw_file)
    if not {"orig_qid", "question"}.issubset(extra.columns):
        raise ValueError("В questions_paraphrased.csv отсутствуют orig_qid или question")
    extra["orig_qid"] = extra["orig_qid"].astype("int64")
    if "qid" not in extra.columns:
        extra["qid"] = np.arange(200_000, 200_000 + len(extra), dtype=np.int64)
    extra["qid"] = extra["qid"].astype("int64")
    extra["is_paraphrase"] = True
    extra["role"] = "train"
    par = pd.concat([par, extra], ignore_index=True, sort=False)

# Сопоставление оригинала и перефраза выполняется ПО orig_qid, не по номеру строки.
par = par.sort_values(["orig_qid", "qid"]).drop_duplicates("orig_qid")
orig_table = train_orig[["qid", "orig_qid", "question"]].copy()
orig_table = orig_table[orig_table.qid == orig_table.orig_qid]
orig_table = orig_table.rename(columns={"qid": "original_qid", "question": "original_question"})
para_table = par[["orig_qid", "qid", "question"]].rename(
    columns={"qid": "paraphrase_qid", "question": "paraphrase_question"}
)
pairs_all = orig_table.merge(para_table, on="orig_qid", how="inner", validate="one_to_one")
pairs_all = pairs_all[
    pairs_all.original_question.astype(str).str.strip().str.casefold() !=
    pairs_all.paraphrase_question.astype(str).str.strip().str.casefold()
].copy()
corpus_ids = set(chunks.chunk_id.astype(int))
strict_gold = defaultdict(set)
soft_gold = defaultdict(set)
for r in answer_chunks.itertuples(index=False):
    oid, cid = int(r.orig_qid), int(r.chunk_id)
    soft_gold[oid].add(cid)
    if bool(r.is_annotated):
        strict_gold[oid].add(cid)
pairs_all = pairs_all[pairs_all.orig_qid.map(
    lambda x: len(strict_gold[int(x)] & corpus_ids) > 0)].copy()
if len(pairs_all) < N_PAIRS:
    raise RuntimeError(
        f"После проверки найдено только {len(pairs_all)} пригодных пар, а нужно {N_PAIRS}. "
        f"Проверьте {DATA_DIR / 'questions.csv'} и questions_paraphrased.csv."
    )
pairs = pd.read_csv(BLOCK1_DIR / 'results/paraphrase/selected_500_pairs.csv')
matched = pairs_all.set_index('orig_qid').reindex(pairs.orig_qid)
for column in ['original_qid', 'original_question', 'paraphrase_qid', 'paraphrase_question']:
    if matched[column].tolist() != pairs[column].tolist():
        raise ValueError('Frozen 500 pairs do not match the corpus: ' + column)
pairs.to_csv(OUTPUT_DIR / "selected_500_pairs.csv", index=False)
query_texts = pairs.original_question.astype(str).tolist() + pairs.paraphrase_question.astype(str).tolist()
query_ids = pairs.original_qid.astype(int).tolist() + pairs.paraphrase_qid.astype(int).tolist()
QUERY_HASH = signature(query_ids, query_texts)
chunk_ids = chunks.chunk_id.to_numpy(dtype=np.int64)
DOC_HASH = signature(chunk_ids, chunks.text.astype(str))
if DOC_HASH != '15704f7b9c5a2a21f5484613e837786e00afc7d596014eb82832dfb0732fb4af':
    raise ValueError('Corpus fingerprint differs from the published experiment; stop and check data')
RUN_HASH = hashlib.sha256((DOC_HASH+QUERY_HASH).encode()).hexdigest()
assert len(query_texts) == 2*N_PAIRS and len(set(pairs.orig_qid)) == N_PAIRS
log(f"Найдено {len(pairs_all)} пригодных пар; отобрано {N_PAIRS}; корпус {len(chunks)} фрагментов")
log("Первый вопрос и перефраз:")
print("Исходный:", pairs.iloc[0].original_question)
print("Перефраз:", pairs.iloc[0].paraphrase_question)
print("Обе формулировки используют одни и те же эталонные chunk_id.")

In [ ]:
# 3. Загружаем СУЩЕСТВУЮЩИЕ документные эмбеддинги, предварительно сверяя метаданные.
def load_document_cache(slug, model_id, expected_dim):
    metadata_file = OLD_RESULTS / "cache" / f"{slug}_docs.json"
    default_path = OLD_RESULTS / "cache" / f"{slug}_docs.npy"
    if not metadata_file.exists():
        raise FileNotFoundError(
            f"Нет {metadata_file}. Не начинаю повторную векторизацию документов. "
            "Сначала нужно восстановить кэш из завершённого BGE-эксперимента."
        )
    metadata = json.loads(metadata_file.read_text(encoding="utf-8"))
    if metadata.get("model_id") != model_id or metadata.get("fingerprint") != DOC_HASH:
        raise ValueError(f"{slug}: модель или контрольная сумма корпуса не совпадает с готовым кэшем")
    path = Path(metadata.get("source", str(default_path)))
    if not path.exists() and default_path.exists():
        path = default_path
    if not path.exists():
        raise FileNotFoundError(f"Файл эмбеддингов {slug} не найден: {path}")
    arr = np.load(path, mmap_mode="r")
    if arr.shape != (len(chunks), expected_dim) or arr.dtype != np.float32:
        raise ValueError(f"Неверная форма {slug}: {arr.shape}; ожидалось {(len(chunks), expected_dim)}")
    indices = np.linspace(0, len(arr)-1, min(50, len(arr))).astype(int)
    samples = np.asarray(arr[indices])
    norms = np.linalg.norm(samples, axis=1)
    if not np.isfinite(samples).all() or np.max(np.abs(norms-1)) > 0.025:
        raise ValueError(f"Нарушена нормировка или обнаружены нечисловые значения в {slug}")
    log(f"[{slug}] готовые документные эмбеддинги: {arr.shape}, файл {path.name}")
    return arr

# Проверяем все 3 кэша до загрузки тяжёлых моделей и кодирования новых вопросов.
for _slug, (_model, _dim) in MODELS.items():
    _temp = load_document_cache(_slug, _model, _dim)
    del _temp
log("Все три документных кэша найдены, корпус и порядок фрагментов совпадают.")

In [ ]:
# 4. Кодируем ТОЛЬКО 1000 запросов каждой BGE с возобновлением по батчам.
def ensure_query_embeddings(slug, model_id, dimension):
    path = OUTPUT_DIR / "cache" / f"{slug}_paired_queries.npy"
    meta_file = OUTPUT_DIR / "cache" / f"{slug}_paired_queries.json"
    key = {"model_id": model_id, "dimension": dimension, "query_hash": QUERY_HASH,
           "query_prefix": QUERY_PREFIX, "version": 1}
    if path.exists() and meta_file.exists():
        try:
            old = json.loads(meta_file.read_text(encoding="utf-8"))
            if old == key:
                cached = np.load(path, mmap_mode="r")
                if cached.shape == (2*N_PAIRS, dimension) and cached.dtype == np.float32:
                    log(f"[{slug}] 1000 вопросов уже закодированы — использую готовый файл")
                    return cached
        except (OSError, ValueError, json.JSONDecodeError):
            pass

    from sentence_transformers import SentenceTransformer
    log(f"[{slug}] Загружаю {model_id}; документы кодироваться НЕ будут")
    model = SentenceTransformer(model_id, device=DEVICE_USED)
    part = path.with_name(path.stem+".partial.npy")
    state = path.with_name(path.stem+".partial.json")
    completed = 0
    data = None
    if part.exists() and state.exists():
        try:
            st = json.loads(state.read_text(encoding="utf-8"))
            data = np.load(part, mmap_mode="r+")
            if st.get("key") == key and data.shape == (2*N_PAIRS, dimension):
                completed = int(st.get("completed", 0))
                if not 0 <= completed <= 2*N_PAIRS:
                    completed = 0
            else:
                del data
                data = None
        except (ValueError, OSError, json.JSONDecodeError):
            data = None
    if data is None or completed == 0:
        if data is not None:
            del data
        data = np.lib.format.open_memmap(part, mode="w+", dtype=np.float32,
                                         shape=(2*N_PAIRS, dimension))
        completed = 0
        atomic_json(state, {"key": key, "completed": 0})
    log(f"[{slug}] вопросы: {completed}/{2*N_PAIRS} уже готовы")
    t0 = time.perf_counter()
    for start in range(completed, 2*N_PAIRS, BATCH_SIZE):
        end = min(start+BATCH_SIZE, 2*N_PAIRS)
        batch = [QUERY_PREFIX+q for q in query_texts[start:end]]
        encoded = model.encode(batch, batch_size=BATCH_SIZE,
                               convert_to_numpy=True, show_progress_bar=False)
        encoded = normalized(encoded)
        if encoded.shape != (end-start, dimension):
            raise RuntimeError(f"{slug}: неожиданный размер результата {encoded.shape}")
        data[start:end] = encoded
        data.flush()
        atomic_json(state, {"key": key, "completed": end})
        if start == completed or end % (BATCH_SIZE*8) == 0 or end == 2*N_PAIRS:
            log(f"[{slug}] кодирование запросов {end}/{2*N_PAIRS}, прошло {time.perf_counter()-t0:.1f} сек")
    del data, model
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
    part.replace(path)
    state.unlink(missing_ok=True)
    atomic_json(meta_file, key)
    return np.load(path, mmap_mode="r")

In [ ]:
# 5. Точный top-k по всему корпусу, с сохранением после каждых 16 вопросов.
def exact_topk_for_pairs(slug, document_vectors, query_vectors):
    top_k = max(K_VALUES)
    result_path = OUTPUT_DIR / "rankings" / f"{slug}_topk_ids.npy"
    state_path = OUTPUT_DIR / "rankings" / f"{slug}_topk_state.json"
    config = {"model": slug, "run_hash": RUN_HASH, "n_query": 2*N_PAIRS,
              "n_doc": len(chunks), "dim": int(document_vectors.shape[1]), "top_k": top_k, "version": 2,
              "vectors_docs": array_hash(document_vectors), "vectors_queries": array_hash(query_vectors)}
    current = 0
    ranking = None
    if result_path.exists() and state_path.exists():
        try:
            st = json.loads(state_path.read_text(encoding="utf-8"))
            ranking = np.load(result_path, mmap_mode="r+")
            if st.get("config") == config and ranking.shape == (2*N_PAIRS, top_k):
                current = int(st.get("completed", 0))
                if not 0 <= current <= 2*N_PAIRS:
                    current = 0
            else:
                del ranking
                ranking = None
        except (OSError, ValueError, json.JSONDecodeError):
            ranking = None
    if ranking is None:
        ranking = np.lib.format.open_memmap(result_path, mode="w+", dtype=np.int64,
                                            shape=(2*N_PAIRS, top_k))
        ranking[:] = -1
        ranking.flush()
        atomic_json(state_path, {"config": config, "completed": 0})
    log(f"[{slug}] точный поиск: {current}/{2*N_PAIRS} вопросов уже обработано")
    t0 = time.perf_counter()
    with threadpool_limits(limits=THREADS):
        for q0 in range(current, 2*N_PAIRS, QUERY_BLOCK):
            q1 = min(q0+QUERY_BLOCK, 2*N_PAIRS)
            q = np.ascontiguousarray(query_vectors[q0:q1], dtype=np.float32)
            count = len(q)
            top_scores = np.full((count, top_k), -np.inf, dtype=np.float32)
            top_ids = np.full((count, top_k), -1, dtype=np.int64)
            for d0 in range(0, len(chunks), DOC_BLOCK):
                d1 = min(d0+DOC_BLOCK, len(chunks))
                docs = np.ascontiguousarray(document_vectors[d0:d1], dtype=np.float32)
                similarities = q @ docs.T
                keep = min(top_k, len(docs))
                order = stable_topk_positions(similarities, chunk_ids[d0:d1], keep)
                new_scores = np.take_along_axis(similarities, order, axis=1)
                new_ids = chunk_ids[d0+order]
                all_scores = np.concatenate([top_scores, new_scores], axis=1)
                all_ids = np.concatenate([top_ids, new_ids], axis=1)
                idx = np.lexsort((all_ids, -all_scores), axis=1)[:, :top_k]
                top_scores = np.take_along_axis(all_scores, idx, axis=1)
                top_ids = np.take_along_axis(all_ids, idx, axis=1)
            ranking[q0:q1] = top_ids
            ranking.flush()
            atomic_json(state_path, {"config": config, "completed": q1})
            if q0 == current or (q1 // QUERY_BLOCK) % 8 == 0 or q1 == 2*N_PAIRS:
                log(f"[{slug}] поиск {q1}/{2*N_PAIRS}, прошло {time.perf_counter()-t0:.1f} сек")
    return np.load(result_path, mmap_mode="r")

# Тестирование математической логики top-k до запуска моделей.
q = normalized([[1,0],[0,1]])
d = normalized([[1,0],[0,1],[-1,0]])
assert np.argmax(q @ d.T, axis=1).tolist() == [0,1]
log("Математическая проверка косинусного поиска пройдена")

In [ ]:
# 6. Запускаем три BGE ПО ОЧЕРЕДИ. После каждой сохраняем результаты (перезапуск безопасен).
all_pair_tables=[]
for slug, (model_id, dimension) in MODELS.items():
    log(f"=== Начинаю {slug} ===")
    docs = load_document_cache(slug, model_id, dimension)
    encoded_queries = ensure_query_embeddings(slug, model_id, dimension)
    retrieved = exact_topk_for_pairs(slug, docs, encoded_queries)
    result = []
    for row_i, p in pairs.iterrows():
        oid = int(p.orig_qid)
        for k in K_VALUES:
            g1 = strict_gold[oid]
            g2 = soft_gold[oid]
            original = set(int(x) for x in retrieved[row_i, :k])
            paraphrase = set(int(x) for x in retrieved[row_i+N_PAIRS, :k])
            result.append({
                "run_hash": RUN_HASH, "model": slug, "dimension": dimension,
                "k": k, "orig_qid": oid, "paraphrase_qid": int(p.paraphrase_qid),
                "strict_original": int(bool(original & g1)),
                "strict_paraphrase": int(bool(paraphrase & g1)),
                "soft_original": int(bool(original & g2)),
                "soft_paraphrase": int(bool(paraphrase & g2)),
                "topk_overlap_fraction": len(original & paraphrase) / k,
            })
    data = pd.DataFrame(result)
    assert len(data) == N_PAIRS*len(K_VALUES)
    assert (data.strict_original <= data.soft_original).all()
    assert (data.strict_paraphrase <= data.soft_paraphrase).all()
    out = OUTPUT_DIR / "tables" / f"{slug}_500pairs.csv"
    data.to_csv(out, index=False)
    all_pair_tables.append(data)
    k5 = data[data.k == 5]
    log(f"[{slug}] ГОТОВО. Hit@5 original={k5.strict_original.mean():.1%}; paraphrase={k5.strict_paraphrase.mean():.1%}; файл: {out.name}")
    del docs, encoded_queries, retrieved
    gc.collect()
log("Три BGE завершены; все исходные эмбеддинги документов остались нетронутыми")

In [ ]:
# 7. Итоговая статистика: только парные сравнения по тем же самым 500 вопросам.
all_pairs = pd.concat(all_pair_tables, ignore_index=True)
all_pairs.to_csv(OUTPUT_DIR / "paraphrase_by_pair.csv", index=False)
rows=[]
pvals_at_5=[]
for (model, k), table in all_pairs.groupby(["model", "k"], sort=True):
    assert len(table) == N_PAIRS
    x = table.strict_original.to_numpy(dtype=int)
    y = table.strict_paraphrase.to_numpy(dtype=int)
    original_hit = float(x.mean()); paraphrase_hit = float(y.mean())
    lost = int(((x==1) & (y==0)).sum())
    gained = int(((x==0) & (y==1)).sum())
    kept = int(((x==1) & (y==1)).sum())
    both_wrong = int(((x==0) & (y==0)).sum())
    # Точный вариант парного теста Мак-Немара (условный биномиальный).
    pval = (float(binomtest(min(lost,gained), n=lost+gained, p=0.5).pvalue)
            if (lost+gained) else 1.0)
    delta = (y-x).astype(np.float64)
    rng = np.random.default_rng(SEED + int(k) + {"bge_small":1,"bge_base":2,"bge_large":3}[model])
    boot = np.empty(5000)
    for b in range(len(boot)):
        draw = rng.integers(0, len(delta), len(delta))
        boot[b] = delta[draw].mean()
    ci_lo, ci_hi = np.quantile(boot, [0.025,0.975])
    rows.append({
        "model": model, "k": int(k), "n_pairs": len(table),
        "strict_original": original_hit, "strict_paraphrase": paraphrase_hit,
        "strict_delta_para_minus_orig": paraphrase_hit-original_hit,
        "strict_delta_ci95_low": float(ci_lo), "strict_delta_ci95_high": float(ci_hi),
        "strict_lost_after_paraphrase": lost, "strict_gained_after_paraphrase": gained,
        "strict_still_correct": kept, "strict_both_incorrect": both_wrong,
        "strict_retention_given_original_correct": kept/(kept+lost) if kept+lost else np.nan,
        "soft_original": float(table.soft_original.mean()),
        "soft_paraphrase": float(table.soft_paraphrase.mean()),
        "soft_delta_para_minus_orig": float((table.soft_paraphrase-table.soft_original).mean()),
        "topk_overlap_mean": float(table.topk_overlap_fraction.mean()),
        "mcnemar_exact_p": pval,
    })
summary = pd.DataFrame(rows)
# Holm correction across the three prespecified primary strict Hit@5 comparisons.
primary = summary[summary.k == 5].copy().sort_values("mcnemar_exact_p")
p_raw = primary.mcnemar_exact_p.to_numpy()
adjusted = np.maximum.accumulate(np.minimum(1.0, p_raw*(len(p_raw)-np.arange(len(p_raw)))))
primary["holm_p_strict_hit5"] = adjusted
summary = summary.merge(primary[["model","holm_p_strict_hit5"]], on="model", how="left")
summary.loc[summary.k != 5, "holm_p_strict_hit5"] = np.nan
summary.to_csv(OUTPUT_DIR / "paraphrase_summary.csv", index=False)
primary.to_csv(OUTPUT_DIR / "paired_tests_hit5.csv", index=False)
print("\n=== ГЛАВНЫЙ РЕЗУЛЬТАТ: Hit@5 на одних и тех же 500 парах ===")
print(primary[["model", "strict_original", "strict_paraphrase",
               "strict_delta_para_minus_orig", "strict_lost_after_paraphrase",
               "strict_gained_after_paraphrase", "mcnemar_exact_p", "holm_p_strict_hit5"]]
      .sort_values("model").to_string(index=False, float_format=lambda v: f"{v:.4f}"))

In [ ]:
# 7a. Прямая проверка исследовательской гипотезы:
#     становится ли модель устойчивее к перефразированию при увеличении её размера?
from itertools import combinations

comparison_rows=[]
for model_a, model_b in combinations(MODELS.keys(), 2):
    a = all_pairs[(all_pairs.model == model_a) & (all_pairs.k == 5)].sort_values('orig_qid')
    b = all_pairs[(all_pairs.model == model_b) & (all_pairs.k == 5)].sort_values('orig_qid')
    if not np.array_equal(a.orig_qid.to_numpy(), b.orig_qid.to_numpy()):
        raise AssertionError('Не совпал набор исходных вопросов для межмодельного сравнения')
    delta_a = (a.strict_paraphrase-a.strict_original).to_numpy(dtype=float)
    delta_b = (b.strict_paraphrase-b.strict_original).to_numpy(dtype=float)
    difference = delta_b-delta_a   # Положительное значение -> модель b устойчивее модели a.
    observed = float(difference.mean())
    rng = np.random.default_rng(SEED+55+len(comparison_rows))
    ci_boot = np.empty(5000)
    for j in range(len(ci_boot)):
        ci_boot[j] = difference[rng.integers(0,len(difference),len(difference))].mean()
    ci_lo,ci_hi = np.quantile(ci_boot,[0.025,0.975])
    # Парный перестановочный тест: случайно меняем метки моделей внутри каждого вопроса.
    # Никакие новые эмбеддинги не вычисляются.
    extreme=0;total=20_000
    for _ in range(20):
        signs = rng.choice([-1,1],size=(1000,len(difference)))
        simulated = (signs*difference).mean(axis=1)
        extreme += int(np.sum(np.abs(simulated)>=abs(observed)-1e-12))
    p_value=(extreme+1)/(total+1)
    comparison_rows.append({'model_a':model_a,'model_b':model_b,'n_pairs':len(difference),
                            'robustness_delta_b_minus_a':observed,
                            'ci95_low':float(ci_lo),'ci95_high':float(ci_hi),
                            'paired_permutation_p':p_value})
comparisons=pd.DataFrame(comparison_rows).sort_values('paired_permutation_p').reset_index(drop=True)
pvals=comparisons.paired_permutation_p.to_numpy(dtype=float)
comparisons['holm_p'] = np.maximum.accumulate(np.minimum(1.0,pvals*(len(pvals)-np.arange(len(pvals)))))
comparisons.to_csv(OUTPUT_DIR/'model_robustness_comparison.csv',index=False)
print('\n=== СРАВНЕНИЕ УСТОЙЧИВОСТИ МЕЖДУ МОДЕЛЯМИ, Strict Hit@5 ===')
print(comparisons.to_string(index=False,float_format=lambda x:f'{x:.4f}'))

In [ ]:
# 8. Графики + краткий отчёт для научной статьи.
plt.rcParams.update({"font.size": 10, "figure.dpi": 130})
fig, ax = plt.subplots(figsize=(8.2, 4.6))
labels = {"bge_small":"BGE Small", "bge_base":"BGE Base", "bge_large":"BGE Large"}
for model in MODELS:
    sub = summary[summary.model == model].sort_values("k")
    ax.plot(sub.k, sub.strict_original*100, marker="o", label=f"{labels[model]} — original")
    ax.plot(sub.k, sub.strict_paraphrase*100, marker="x", linestyle="--", label=f"{labels[model]} — paraphrase")
ax.set_xticks(list(K_VALUES))
ax.set_xlabel("k: число извлечённых фрагментов")
ax.set_ylabel("Strict Hit@k, %")
ax.set_title("Устойчивость BGE к перефразированию (500 пар, полный SQuAD)")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "figures" / "hitk_original_vs_paraphrase.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7.2, 4.2))
plot_data = primary.set_index("model").reindex(list(MODELS))
heights = 100*plot_data.strict_delta_para_minus_orig
ax.bar([labels[x] for x in plot_data.index], heights)
ax.axhline(0, linewidth=1)
ax.set_ylabel("Δ Hit@5 (перефраз − оригинал), п. п.")
ax.set_title("Изменение качества после перефразирования")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "figures" / "paraphrase_change_hit5.png", bbox_inches="tight")
plt.show()

report = {
    "purpose": "Paired paraphrase robustness for BGE small/base/large",
    "source": "Lossy-RAG SQuAD v1.1 train and repository paraphrases",
    "n_chunks": len(chunks), "n_pairs": int(N_PAIRS),
    "doc_fingerprint": DOC_HASH, "question_fingerprint": QUERY_HASH,
    "seed": SEED, "k_values": list(K_VALUES), "models": list(MODELS),
    "search": "exact cosine similarity on cached float32 document embeddings",
    "gold": {"strict": "is_annotated=true", "soft": "all answer_chunks"},
    "test": "paired exact McNemar; Holm correction across 3 primary strict Hit@5 comparisons",
    "caveat": "Paraphrases are taken from the project's dataset. Their semantic equivalence is assumed, not independently human-validated.",
    "completed_at": datetime.now().isoformat(),
}
atomic_json(OUTPUT_DIR / "paraphrase_report.json", report)
log("ВСЁ ГОТОВО. Главное: paraphrase_summary.csv и paired_tests_hit5.csv")
print("\nОтправь в чат эти два файла:")
print(OUTPUT_DIR / "paraphrase_summary.csv")
print(OUTPUT_DIR / "paired_tests_hit5.csv")
print(OUTPUT_DIR / "model_robustness_comparison.csv")

## Как интерпретировать результаты

В `paraphrase_summary.csv`:

- `strict_original` — доля найденных правильных фрагментов при исходной формулировке.
- `strict_paraphrase` — та же метрика при изменённой формулировке **на тех же 500 вопросах**.
- `strict_delta_para_minus_orig` — разница (отрицательное значение означает ухудшение).
- `strict_lost_after_paraphrase` — сколько из ранее успешных вопросов перестали находиться.
- `strict_gained_after_paraphrase` — сколько, напротив, стали находиться.
- `strict_retention_given_original_correct` — какая доля исходно успешных запросов осталась успешной.
- `mcnemar_exact_p` — парный точный тест; в `paired_tests_hit5.csv` есть также `holm_p_strict_hit5`, учитывающий три сравниваемые модели.

**Итог:** пришли `paraphrase_summary.csv`, `paired_tests_hit5.csv` и `model_robustness_comparison.csv`. Для анализа ошибок можно добавить `paraphrase_by_pair.csv` и `selected_500_pairs.csv`. Никакие ранее полученные BGE/PCA/Nomic результаты переписывать или пересчитывать не нужно.